In [ ]:
import os
import pandas as pd
import numpy as np
from bids import BIDSLayout
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# 1. Setup Layouts
raw_data_path = '/Volumes/T9/ds001486'
deriv_path = os.path.join(raw_data_path, 'derivatives/fmriprep')
layout_deriv = BIDSLayout(deriv_path, validate=False, derivatives=False)

# Get ONLY the Subtraction BOLD files
bold_files = layout_deriv.get(task='Sub', suffix='bold', extension='nii.gz', desc='preproc', return_type='file')

mld_subs = ['059', '065', '067', '069', '071', '075', '076', '077', '078', '083', '088', '095', '096', '103', '106']
confounds_of_interest = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z', 'csf', 'white_matter']

# 2. Setup the Masker (Schaefer 100 Regions)
schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)
masker = NiftiLabelsMasker(labels_img=schaefer.maps, standardize=True, detrend=True)

# 3. Setup the Connectivity Measure
# vectorize=True flattens the matrix. discard_diagonal=True removes the useless 1s.
connectome_measure = ConnectivityMeasure(kind='correlation', vectorize=True, discard_diagonal=True)

time_series_list = []
y_sub = []

print(f"Extracting 4D Time-Series for {len(bold_files)} runs. This may take a few minutes...")

for bold_path in bold_files:
    entities = layout_deriv.parse_file_entities(bold_path)
    sub_id = entities.get('subject')
    ses_id = entities.get('session')
    run_id = entities.get('run')

    # Locate Confounds
    confound_files = layout_deriv.get(subject=sub_id, session=ses_id, run=run_id, desc='confounds', extension='tsv', return_type='file')
    
    if not confound_files:
        continue
        
    try:
        # Load and filter confounds
        confounds_df = pd.read_csv(confound_files[0], sep='\t')
        avail_confounds = [c for c in confounds_of_interest if c in confounds_df.columns]
        selected_confounds = confounds_df[avail_confounds].fillna(0).values

        # Extract the time-series for the 100 regions (Cleaned using confounds)
        time_series = masker.fit_transform(bold_path, confounds=selected_confounds)
        time_series_list.append(time_series)
        
        # Build the label
        y_sub.append(1 if sub_id in mld_subs else 0)
        
    except Exception as e:
        print(f"Error on sub-{sub_id}: {e}")

# 4. Calculate Connectivity Matrices
# This turns the time-series into 4,950 unique connections per subject
print("Calculating Functional Connectivity Matrices...")
X_sub = connectome_measure.fit_transform(time_series_list)
y_sub = np.array(y_sub)

print(f"Final Feature Matrix X shape: {X_sub.shape} | Labels y shape: {y_sub.shape}\n")

# 5. Machine Learning Evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "SVM (RBF Kernel)": SVC(kernel='rbf', random_state=42),
    "SVM (Linear)": SVC(kernel='linear', random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42)
}

print("--- 5-Fold CV Accuracy for FUNCTIONAL CONNECTIVITY (Subtraction) ---")
for name, model in models.items():
    pipeline = make_pipeline(StandardScaler(), model)
    scores = cross_val_score(pipeline, X_sub, y_sub, cv=cv, scoring='accuracy')
    print(f"{name:20s}: {scores.mean():.3f} (± {scores.std():.3f})")

[fetch_atlas_schaefer_2018] Dataset found in /Users/jchong058/nilearn_data/schaefer_2018
Extracting 4D Time-Series for 120 Subtraction runs. This may take a few minutes...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_54098/649504690.py:64: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_54098/649504690.py:64: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_54098/649504690.py:64: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_54098/649504690.py:64: DeprecationWarning: From release 0.14.0, confounds will be standar

Calculating Functional Connectivity Matrices...
Final Feature Matrix X shape: (120, 4950) | Labels y shape: (120,)

--- 5-Fold CV Accuracy for FUNCTIONAL CONNECTIVITY (Subtraction) ---
SVM (RBF Kernel)    : 0.817 (± 0.042)
SVM (Linear)        : 0.942 (± 0.033)
XGBoost             : 0.692 (± 0.077)
Random Forest       : 0.692 (± 0.094)
Logistic Regression : 0.933 (± 0.033)


In [ ]:
import os
import pandas as pd
import numpy as np
import nibabel as nib
from bids import BIDSLayout
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# 1. Setup Layouts
raw_data_path = '/Volumes/T9/ds001486'
deriv_path = os.path.join(raw_data_path, 'derivatives/fmriprep')
layout_deriv = BIDSLayout(deriv_path, validate=False, derivatives=False)

bold_files = layout_deriv.get(task='Mult', suffix='bold', extension='nii.gz', desc='preproc', return_type='file')

mld_subs = ['059', '065', '067', '069', '071', '075', '076', '077', '078', '083', '088', '095', '096', '103', '106']
confounds_of_interest = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z', 'csf', 'white_matter']

# 2. Setup the Masker (FIXED WARNING: standardize='zscore_sample')
schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)
masker = NiftiLabelsMasker(labels_img=schaefer.maps, standardize='zscore_sample', detrend=True)

# 3. Setup the Connectivity Measure
connectome_measure = ConnectivityMeasure(kind='correlation', vectorize=True, discard_diagonal=True)

time_series_list = []
y_sub = []

print(f"Extracting 4D Time-Series for {len(bold_files)} Subtraction runs. This may take a few minutes...")

for bold_path in bold_files:
    entities = layout_deriv.parse_file_entities(bold_path)
    sub_id = entities.get('subject')
    ses_id = entities.get('session')
    run_id = entities.get('run')

    # Locate Confounds
    confound_files = layout_deriv.get(subject=sub_id, session=ses_id, task='Sub', run=run_id, desc='confounds', extension='tsv', return_type='file')
    
    if not confound_files:
        continue
        
    try:
        # FIXED ERROR: Find exact length of the BOLD image
        img = nib.load(bold_path)
        n_vols = img.shape[3]

        # Load and filter confounds
        confounds_df = pd.read_csv(confound_files[0], sep='\t')
        avail_confounds = [c for c in confounds_of_interest if c in confounds_df.columns]
        
        # Slice the confounds array to perfectly match the number of BOLD volumes
        selected_confounds = confounds_df[avail_confounds].fillna(0).values[:n_vols, :]

        # Extract the time-series for the 100 regions
        time_series = masker.fit_transform(bold_path, confounds=selected_confounds)
        time_series_list.append(time_series)
        
        # Build the label
        y_sub.append(1 if sub_id in mld_subs else 0)
        
    except Exception as e:
        print(f"Error on sub-{sub_id}: {e}")

# 4. Calculate Connectivity Matrices
print("Calculating Functional Connectivity Matrices...")
X_sub = connectome_measure.fit_transform(time_series_list)
y_sub = np.array(y_sub)

print(f"Final Feature Matrix X shape: {X_sub.shape} | Labels y shape: {y_sub.shape}\n")

# 5. Machine Learning Evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "SVM (RBF Kernel)": SVC(kernel='rbf', random_state=42),
    "SVM (Linear)": SVC(kernel='linear', random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42)
}

print("--- 5-Fold CV Accuracy for FUNCTIONAL CONNECTIVITY (Subtraction) ---")
for name, model in models.items():
    pipeline = make_pipeline(StandardScaler(), model)
    scores = cross_val_score(pipeline, X_sub, y_sub, cv=cv, scoring='accuracy')
    print(f"{name:20s}: {scores.mean():.3f} (± {scores.std():.3f})")

[fetch_atlas_schaefer_2018] Dataset found in /Users/jchong058/nilearn_data/schaefer_2018
Extracting 4D Time-Series for 120 Subtraction runs. This may take a few minutes...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_54098/649504690.py:64: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_54098/649504690.py:64: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_54098/649504690.py:64: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  time_series = masker.fit_transform(bold_path, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_54098/649504690.py:64: DeprecationWarning: From release 0.14.0, confounds will be standar

Calculating Functional Connectivity Matrices...
Final Feature Matrix X shape: (120, 4950) | Labels y shape: (120,)

--- 5-Fold CV Accuracy for FUNCTIONAL CONNECTIVITY (Subtraction) ---
SVM (RBF Kernel)    : 0.817 (± 0.042)
SVM (Linear)        : 0.942 (± 0.033)
XGBoost             : 0.692 (± 0.077)
Random Forest       : 0.692 (± 0.094)
Logistic Regression : 0.933 (± 0.033)
